# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the [FAIR² Mental Health Survey Dataset](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library, referencing all dataset entities using their `@id`.

### Dataset Source
The dataset is described by a Croissant schema at:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json


In [ ]:
# Ensure mlcroissant is installed in your environment
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview

Review available record sets and their fields using their `@id`. 

This dataset may contain multiple record sets; we list them along with field IDs for reference.

In [ ]:
# List all record sets with their @id and fields referenced by @id
record_sets = list(dataset.record_sets)
print(f"Record sets discovered (showing @id and name):\n")
for rs in record_sets:
    name = getattr(rs, 'name', None)
    print(f" - @id: {rs.id}, name: {name}")
    print("   Fields in this record set, referenced by @id:")
    for field in getattr(rs, 'fields', []):
        print(f"     - {field.id}")
    print()
# Show an example record from the first record set
if record_sets:
    example_records = list(dataset.records(record_set=record_sets[0].id))
    print(f"Example record from {record_sets[0].id}:")
    if example_records:
        print(example_records[0])
    else:
        print("No records available.")

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames for analysis using the record set and field `@id`s identified above.

In [ ]:
# Prepare to extract data for all record sets
from collections import OrderedDict

all_record_set_ids = [rs.id for rs in record_sets]
dataframes = OrderedDict()
print(f"Extracting the following record sets by @id: {all_record_set_ids}\n")

for rs in record_sets:
    # Load as DataFrame
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded record set {rs.id} with shape {df.shape}")
    if df.shape[0] > 0:
        print(f"Columns (@id): {list(df.columns)}\n")
    else:
        print("No records found in this record set.\n")

# Pick the primary record set (often the first one)
if all_record_set_ids:
    MAIN_RECORD_SET_ID = all_record_set_ids[0]
    print(f"== Sample data from the main record set (@id: {MAIN_RECORD_SET_ID}): ==")
    display(dataframes[MAIN_RECORD_SET_ID].head())

## 4. Exploratory Data Analysis (EDA)

We'll process and examine the main record set, demonstrating filtering, normalization, and grouping by field `@id` as recommended.

**Note**: Update the variables below—the field `@id`s used for demonstration come from the output above.

In [ ]:
# Select numeric and group field by their @id.
# Replace <numeric_field_id> and <group_field_id> with actual field ids from previous cell output.
numeric_field_id = None
group_field_id = None
df = dataframes[MAIN_RECORD_SET_ID]

# Try to auto detect common mental health assessment fields for demonstration (e.g., GAD-7, PHQ-9, PSQ total scores)
for col in df.columns:
    if 'gad_7_score' in col.lower():
        numeric_field_id = col
    elif 'phq_9_score' in col.lower():
        numeric_field_id = col
    elif 'psq_score' in col.lower():
        numeric_field_id = col
    if 'gender' in col.lower():
        group_field_id = col
    elif 'age_group' in col.lower():
        group_field_id = col

if numeric_field_id is None and len(df.columns) > 0:
    # Default to first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if group_field_id is None and len(df.columns) > 1:
    group_field_id = df.columns[1]

print(f"Using numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}")

# Drop records with missing numeric field values
eda_df = df.copy()
eda_df = eda_df[pd.notnull(eda_df[numeric_field_id])]

# Example: filter for high scores (threshold=10)
threshold = 10
filtered_df = eda_df[eda_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} found.")
display(filtered_df.head())

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Sample normalized values for {numeric_field_id}:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Group by group_field (e.g., gender or age group) if present
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped {numeric_field_id} mean by {group_field_id}:")
    display(grouped.head())

## 5. Visualization

Visualize distributions, group differences, or correlations in the main record set. Update the code to match actual field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If group field is available, show boxplot
if group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

This notebook demonstrated how to load, explore, and analyze the [FAIR² Mental Health Survey Dataset](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

- We referenced all record sets and fields exclusively via their `@id`s, following best practices for reproducibility.
- Through basic exploratory analysis and visualization, we revealed key characteristics and distributions of the dataset, including a demonstration with a mental health score and grouping by a demographic field.

Explore the dataset further by selecting other fields or conducting in-depth analysis as required for your research context.